**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 25 · 项目收尾：仓库、README 与局限

**心理大模型评测与 Agent** · 项目实战 · 建议用时 4 到 5 小时

**今天的目标**

- 把项目整理成一条命令可复现的仓库
- 写出包含结果和区间的 README
- 写出具体的局限，而不是泛泛的“样本量小”

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 用结果文件自动填 README

读取第 20 天生成的 `results/`（如果没有，就用模拟数据现算），生成 README 草稿，保证文中数字与结果文件一致。

In [ ]:
import pathlib
R20 = pathlib.Path("../day20_project_analysis/results")
if (R20 / "rq2_validity.csv").exists():
    rq1 = pd.read_csv(R20 / "rq1_reliability.csv", index_col=0)
    rq2 = pd.read_csv(R20 / "rq2_validity.csv", index_col=0)
    source = "第 20 天的结果文件"
else:
    from course.data import simulate_counselor_study, DIMENSIONS
    from course.agreement import krippendorff_alpha
    st = simulate_counselor_study(seed=20); h = st["human"]
    rq1 = pd.DataFrame({"alpha_counselors": [krippendorff_alpha(h[h.dim == d].pivot_table(index=["item", "system"], columns="rater", values="score").values.T, "ordinal") for d in DIMENSIONS]}, index=DIMENSIONS)
    rq2 = pd.DataFrame({"kappa": [np.nan] * 5, "lo": [np.nan] * 5, "hi": [np.nan] * 5}, index=DIMENSIONS)
    source = "模拟数据(请先运行第 20 天)"
print("来源:", source)

In [ ]:
def fmt_row(d):
    k = rq2.loc[d]
    ci = "未计算" if pd.isna(k["kappa"]) else f"{k['kappa']:.2f} [{k['lo']:.2f}, {k['hi']:.2f}]"
    return f"| {d} | {rq1.loc[d, 'alpha_counselors']:.2f} | {ci} |"

readme = "\n".join([
    "# LLM 裁判评价 CBT 取向心理支持回复的效度",
    "",
    "**研究问题**:LLM 裁判能否替代咨询师评价心理支持回复?在哪些维度、哪些人群上不能?",
    "",
    "## 如何运行",
    "```bash", "pip install -r requirements.txt", "python run_all.py", "```",
    "",
    "## 主要结果",
    "",
    "| 维度 | 咨询师间 α | 裁判与咨询师 κ [95% CI] |",
    "|---|---|---|",
    *[fmt_row(d) for d in rq1.index],
    "",
    f"数据来源:{source}。",
    "",
    "## 局限",
    "",
    "- (具体写出:影响哪个结论,往哪个方向偏)",
    "",
    "## 数据与伦理",
    "",
    "所有题目为研究者编写,不含真实来访者文字。评分者姓名不公开。",
])
pathlib.Path("README_draft.md").write_text(readme, encoding="utf-8")
print(readme)

**练习 25.1**:README 草稿里应当出现“95%”和“局限”。检查一下。

In [ ]:
ans_25_1 = readme
check_true("练习 25.1", None if ans_25_1 is None else ("95%" in ans_25_1 and "局限" in ans_25_1))

## 2. run_all.py 模板

In [ ]:
run_all = '''"""Reproduce every table and figure: python run_all.py"""
import json, pathlib, subprocess, sys

STEPS = ["src/generate_replies.py", "src/judge_scores.py", "src/analysis.py"]
for step in STEPS:
    print("running", step)
    subprocess.run([sys.executable, step], check=True)
print("done; see results/")
'''
pathlib.Path("run_all_template.py").write_text(run_all, encoding="utf-8")
print(run_all)

**练习 25.2**：写出你的项目的三条具体局限（每条一句，说明影响哪个结论、往哪个方向偏），存成列表 `ans_25_2`。

In [ ]:
ans_25_2 = ["只有 3 位咨询师,门槛判断在区间边缘的维度上不稳定", "裁判与系统 B 同源,自我偏好可能被高估或与长度混淆", "只用了英文题目,结论不能直接推广到中文老年人场景"]
check_true("练习 25.2 三条具体局限", None if ans_25_2 is None else (isinstance(ans_25_2, list) and len(ans_25_2) >= 3))

## 口述题（录音，每题 90 秒）

1. 你的项目最大的局限是什么？
   <details><summary>参考要点</summary>具体回答：例如评分者人数、题目数量、裁判同源、只用了英文题目。说明它影响哪个结论、往哪个方向偏，以及下一步怎么补。</details>

In [ ]:
summary()